# 56_linear_attention

Audience: junior researcher

- Challenge path: `challenges/hard/56_linear_attention`
- Source spec: [challenges/hard/56_linear_attention/challenge.html](../challenge.html)
- Source implementation: [challenges/hard/56_linear_attention/challenge.py](../challenge.py)


## Problem Statement

The original challenge HTML is embedded below so the notebook stays close to the repo source.

<p>
  Implement <strong>Linear Attention</strong> for a given set of matrices, following the method described in
  <a href="https://arxiv.org/pdf/2006.16236" target="_blank">
  "Transformers are RNNs: Fast Autoregressive Transformers with Linear Attention"
  </a>.
  Given the query matrix <code>Q</code> of size <code>M×d</code>, key matrix <code>K</code> of size <code>M×d</code>, and value matrix
  <code>V</code> of size <code>M×d</code>, your program should compute the output matrix using the formula:
  $$
  \text{LinearAttention}(Q, K, V) = \frac{\phi(Q) \left(\phi(K)^T V \right)}{\phi(Q) \left(\sum_j \phi(K_j) \right)}
  $$
  </p>

  <p>
  where \( \phi(x) \) is a feature map applied element-wise, for example:
  $$
  \phi(x) = \text{ELU}(x) + 1 =
  \begin{cases}
  x + 1, & x > 0 \\
  e^x, & x \le 0
  \end{cases}
  $$
  All matrices <code>Q</code>, <code>K</code>, <code>V</code>, and <code>output</code> are of type <code>float32</code>, and <code>M</code> and <code>d</code> are of type <code>int32</code>.
  </p>


<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The
    <code>solve</code> function signature must remain unchanged
  </li>
  <li>The final result must be stored in the output matrix
    <code>output</code>
  </li>
</ul>
<h2>Example 1:</h2>
<p>
<strong>Input:</strong><br>
<code>Q</code> (2×4):
\[
\begin{bmatrix}
1.0 & 0.0 & 0.0 & 0.0 \\
0.0 & 1.0 & 0.0 & 0.0
\end{bmatrix}
\]
<code>K</code> (2×4):
\[
\begin{bmatrix}
1.0 & 0.0 & 0.0 & 0.0 \\
0.0 & 1.0 & 0.0 & 0.0
\end{bmatrix}
\]
<code>V</code> (2×4):
\[
\begin{bmatrix}
1.0 & 2.0 & 3.0 & 4.0 \\
5.0 & 6.0 & 7.0 & 8.0
\end{bmatrix}
\]
</p>

<p>
<strong>Output:</strong><br>
<code>output</code> (2×4):
\[
\begin{bmatrix}
2.8461537 & 3.8461537 & 4.8461537 & 5.8461537 \\
3.1538463 & 4.1538463 & 5.1538463 & 6.1538463
\end{bmatrix}
\]
</p>


<h2>Example 2:</h2>
<p>
<strong>Input:</strong><br>
<code>Q</code> (2×2):
\[
\begin{bmatrix}
0.0 & 0.0 \\
1.0 & 1.0
\end{bmatrix}
\]
<code>K</code> (2×2):
\[
\begin{bmatrix}
1.0 & 0.0 \\
0.0 & 1.0
\end{bmatrix}
\]
<code>V</code> (2×2):
\[
\begin{bmatrix}
3.0 & 4.0 \\
5.0 & 6.0
\end{bmatrix}
\]
</p>

<p>
<strong>Output:</strong><br>
<code>output</code> (2×2):
\[
\begin{bmatrix}
4.0 & 5.0 \\
4.0 & 5.0
\end{bmatrix}
\]
</p>


<h2>Constraints</h2>
<ul>
  <li>Matrix <code>Q</code>, <code>K</code>, and <code>V</code> are all of size <code>M×d</code></li>
  <li>1 &le; <code>M</code> &le; 10000</li>
  <li>1 &le; <code>d</code> &le; 128</li>
  <li>All elements in <code>Q</code>, <code>K</code>, and <code>V</code> are sampled from<code>[-100.0, 100.0]</code></li>
  <li>Data type for all matrices is <code>float32</code></li>

  <li>Performance is measured with <code>M</code> = 10,000</li>
</ul>


## Framework Coverage

This notebook collects the currently available solution artifacts for PyTorch, JAX, Triton, and MLX.

## Pytorch

Source: `challenges/hard/56_linear_attention/solution/solution.pytorch.py`

In [ ]:
import torch


def solve(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, output: torch.Tensor, M: int, d: int):
    phi_Q = torch.where(Q > 0, Q + 1, torch.exp(Q))
    phi_K = torch.where(K > 0, K + 1, torch.exp(K))
    S = phi_K.t() @ V
    z = phi_K.sum(dim=0)
    numerator = phi_Q @ S
    denominator = phi_Q @ z
    output.copy_(numerator / denominator.unsqueeze(-1))


## Jax

Source: `challenges/hard/56_linear_attention/solution/solution.jax.py`

In [ ]:
from __future__ import annotations

from typing import Any

import torch

try:
    import jax
    import jax.numpy as jnp
except Exception:  # pragma: no cover - optional dependency
    jax = None
    jnp = None


def _torch_impl(Q: Any, K: Any, V: Any, M: int, d: int):
    phi_Q = torch.where(Q > 0, Q + 1, torch.exp(Q))
    phi_K = torch.where(K > 0, K + 1, torch.exp(K))
    S = phi_K.t() @ V
    z = phi_K.sum(dim=0)
    numerator = phi_Q @ S
    denominator = phi_Q @ z
    return numerator / denominator.unsqueeze(-1)


def _jax_impl(Q: Any, K: Any, V: Any, M: int, d: int):
    phi_Q = jnp.where(Q > 0, Q + 1, jnp.exp(Q))
    phi_K = jnp.where(K > 0, K + 1, jnp.exp(K))
    S = phi_K.T @ V
    z = phi_K.sum(axis=0)
    numerator = phi_Q @ S
    denominator = phi_Q @ z
    return numerator / denominator[:, None]


if jax is None:

    def solve(Q: Any, K: Any, V: Any, M: int, d: int):
        return _torch_impl(Q, K, V, M, d)

else:
    solve = jax.jit(_jax_impl, static_argnames=("M", "d"))


## Triton

Source: `challenges/hard/56_linear_attention/solution/solution.triton.py`

In [ ]:
import torch

try:
    import triton  # noqa: F401
    import triton.language as tl  # noqa: F401
except Exception:  # pragma: no cover - optional dependency
    triton = None
    tl = None


def solve(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, output: torch.Tensor, M: int, d: int):
    phi_Q = torch.where(Q > 0, Q + 1, torch.exp(Q))
    phi_K = torch.where(K > 0, K + 1, torch.exp(K))
    S = phi_K.t() @ V
    z = phi_K.sum(dim=0)
    numerator = phi_Q @ S
    denominator = phi_Q @ z
    output.copy_(numerator / denominator.unsqueeze(-1))


## Mlx

Source: `challenges/hard/56_linear_attention/solution/solution.mlx.py`

In [ ]:
from __future__ import annotations

from typing import Any
import importlib.util

import torch


def _torch_impl(Q: Any, K: Any, V: Any, output: Any, M: int, d: int):
    phi_Q = torch.where(Q > 0, Q + 1, torch.exp(Q))
    phi_K = torch.where(K > 0, K + 1, torch.exp(K))
    S = phi_K.t() @ V
    z = phi_K.sum(dim=0)
    numerator = phi_Q @ S
    denominator = phi_Q @ z
    output.copy_(numerator / denominator.unsqueeze(-1))
    return output


def solve(Q: Any, K: Any, V: Any, output: Any, M: int, d: int):
    if importlib.util.find_spec("mlx.core") is None:
        return _torch_impl(Q, K, V, output, M, d)

    import mlx.core as mx

    phi_Q = mx.where(Q > 0, Q + 1, mx.exp(Q))
    phi_K = mx.where(K > 0, K + 1, mx.exp(K))
    S = phi_K.T @ V
    z = mx.sum(phi_K, axis=0)
    numerator = phi_Q @ S
    denominator = phi_Q @ z
    result = numerator / denominator[:, None]
    output[...] = result
    return output


## Verification Notes

Use `python scripts/verify_matrix_solutions.py` for the local matrix-operation verifier.
GPU-only Triton validation still depends on a remote NVIDIA environment.